# Lesson 2.1 — experiments: the search tree and the state space

Run the cells in order. Before each experiment, write down your prediction in the "Your prediction" cell (double-click it to edit), and only then run the code.

The first cell chooses the implementation:

- `"mine"` uses your `task.py`; solve Part B of the task first;
- `"reference"` uses the course's reference implementation. Set `IMPLEMENTATION = "reference"` if you want to run the notebook before finishing the task.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

# "mine" uses your task.py; "reference" uses the course's reference implementation.
IMPLEMENTATION = os.environ.get("INTROAI_IMPLEMENTATION", "mine")
if IMPLEMENTATION == "mine":
    import task as algorithms
else:
    from introai.search import reference as algorithms

The function `levels(problem, depth)` builds the search tree level by level with `expand`, without remembering states. The function `growth(problem, depth)` returns, for each depth, the number of tree nodes (from `count_tree_nodes`), the number of different states at that depth, and the number of all states reached up to and including that depth. The function `show` prints these numbers as a table, and `plot_growth` draws them.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from introai.search import Node
from introai.world.plot import plot_map
from introai.world.warehouse import RouteProblem, WarehouseMap, render_text

MAPS = Path.cwd().parents[2] / "data" / "maps"
SERIES = ((1, "#2a78d6", "tree nodes"), (2, "#eb6834", "distinct states"))


def levels(problem, depth):
    """The search tree level by level (lists of nodes), built with expand."""
    tree = [[Node(problem.initial)]]
    for _ in range(depth):
        tree.append([child for node in tree[-1] for child in algorithms.expand(problem, node)])
    return tree


def growth(problem, depth):
    """Rows of (depth, tree nodes, states at that depth, states reached so far)."""
    counts = list(algorithms.count_tree_nodes(problem, depth))
    rows, reached = [], set()
    for d, level in enumerate(levels(problem, depth)):
        if d >= len(counts) or counts[d] != len(level):
            print(f"warning: count_tree_nodes and expand disagree at depth {d} - check task.py")
        states = {node.state for node in level}
        reached |= states
        rows.append((d, counts[d] if d < len(counts) else len(level), len(states), len(reached)))
    return rows


def show(rows):
    print(f"{'depth':>5} {'tree nodes':>11} {'states':>7} {'reached':>8}")
    for d, nodes, states, reached in rows:
        print(f"{d:>5} {nodes:>11,} {states:>7} {reached:>8}")


def plot_growth(rows, title, ax):
    """Tree nodes and distinct states per depth, on a logarithmic scale."""
    depths = [row[0] for row in rows]
    for column, color, label in SERIES:
        values = [row[column] for row in rows]
        ax.plot(
            depths,
            values,
            color=color,
            linewidth=2,
            marker="o",
            markersize=8,
            markeredgecolor="#fcfcfb",
            markeredgewidth=1.5,
            label=label,
        )
        ax.annotate(
            f"{values[-1]:,}",
            (depths[-1], values[-1]),
            textcoords="offset points",
            xytext=(8, 0),
            va="center",
            color="#0b0b0b",
        )
    ax.set_yscale("log")
    ax.set_title(title, loc="left", color="#0b0b0b")
    ax.set_xlabel("depth", color="#52514e")
    ax.set_ylabel("count (log scale)", color="#52514e")
    ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#c3c2b7")
    ax.tick_params(colors="#52514e")
    ax.margins(x=0.12)
    ax.legend(frameon=False, loc="upper left", labelcolor="#0b0b0b")

## Experiment 1: the map with aisles

The robot `R` stands in the top corridor. We build the search tree down to depth 12, that is all routes of up to 12 steps.

In [ ]:
aisles = WarehouseMap.from_file(MAPS / "aisles.txt")
print(render_text(aisles))

**Your prediction.** How many tree nodes lie at depth 12? How many different cells are among them? The order of magnitude is enough.

*My prediction:* …

In [ ]:
rows = growth(RouteProblem(aisles), 12)
show(rows)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
plot_growth(rows, "Search tree and state space, aisles.txt", ax)
fig.tight_layout()

**Think about it.** Compare the result with your prediction. The chart uses a logarithmic scale: exponential growth looks like a straight line on it. Roughly how many times does the number of tree nodes grow with each level? What does that mean for an algorithm that does not remember the states it has visited?

## Experiment 2: where the robot can be after 11 and after 12 steps

In both pictures the blue cells are the states of the tree nodes at that depth.

**Your prediction.** Can the cells reachable in exactly 11 steps and in exactly 12 steps overlap? Why does the number of different states in the table sometimes grow and sometimes drop?

*My prediction:* …

In [ ]:
tree = levels(RouteProblem(aisles), 12)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, depth in zip(axes, (11, 12)):
    cells = {node.state for node in tree[depth]}
    plot_map(
        aisles,
        expanded=cells,
        title=f"states at depth {depth}",
        ax=ax,
        expanded_label="state at this depth",
    )
fig.tight_layout()

**Think about it.** Imagine the map painted like a chessboard. Which squares does every move lead to? What does that mean for the states at even and at odd depths? The `reached` column in the table, on the other hand, grows all the time: how many of those states does a graph search need?

## Experiment 3: the open floor

The map `open_floor` has almost no obstacles, so from most cells the robot can drive in four directions. To save time we only go down to depth 8.

**Your prediction.** How many tree nodes will there be at depth 8, compared with the map with aisles? And how many different states?

*My prediction:* …

In [ ]:
open_floor = WarehouseMap.from_file(MAPS / "open_floor.txt")
open_rows = growth(RouteProblem(open_floor), 8)
show(open_rows)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
plot_growth(rows[:9], "aisles.txt", axes[0])
plot_growth(open_rows, "open_floor.txt", axes[1])
fig.tight_layout()

**Think about it.** How does the branching factor affect the slope of the blue line? How many times more tree nodes are there on the open floor, and how many times more different states? What does a larger branching factor multiply above all? How many expansions would the BFS of lesson 2.2 need to explore all cells down to depth 8?

## Summary

Go back to the review questions in the lesson. After your own attempt, you can compare your results and conclusions with the answers in `solutions/02_search/01_state_space/answers.md`.